# Upload the existing CSV and run the matched benchmark
CPU runtime is sufficient. Select Run all, then upload current_dataset.csv when prompted. No WRDS login, repository upload, or embedded payload is needed.
The CSV contains the predownloaded AMD/NVDA panel. The trainer builds the seven features and compares the nine-size neural policy with the seven-feature tree and original forecast NN.
Attach the downloaded neural_policy_results.zip to our chat. This covers AMD/NVDA only. Keep the Colab runtime connected; resets lose progress.


In [ ]:
%pip install -q numpy==2.2.5 pandas==2.3.3 scikit-learn==1.7.2
import os
os.chdir('/content')
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, io
import pandas as pd

print('Upload current_dataset.csv from the project data folder.')
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_files) != 1:
    raise ValueError('Upload exactly one CSV: current_dataset.csv')
blob = uploaded[csv_files[0]]
panel = pd.read_csv(io.BytesIO(blob))
required = ['date', 'end', 'spread', 'amd_earnings_days_capped',
            'nvda_earnings_days_capped', 'trailing_rv_amd', 'trailing_rv_nvda',
            'iv_amd', 'iv_nvda', 'rv_amd', 'rv_nvda']
missing = sorted(set(required) - set(panel.columns))
if missing:
    raise ValueError('Wrong input CSV; missing columns: ' + ', '.join(missing))
destination = Path('data/current_dataset.csv')
destination.parent.mkdir(exist_ok=True)
destination.write_bytes(blob)
destination.with_suffix('.provenance.json').write_text(json.dumps({
    'source': 'User-uploaded predownloaded AMD/NVDA panel',
    'sha256': hashlib.sha256(blob).hexdigest(), 'rows': len(panel),
    'first': str(panel.date.min()), 'last': str(panel.date.max()),
    'earnings_assumption': 'Historical earnings countdowns from original cached benchmark.'
}, indent=2))
print(f'Ready: {len(panel):,} rows, {panel.date.min()} through {panel.date.max()}')


In [ ]:
%%writefile /content/train_neural.py
import argparse
import hashlib
import json
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
ACTIONS = np.array([-1, 0, 1])
DEFAULT_EXECUTION_COST = 1.0
SHARPE_ANNUALIZATION = np.sqrt(365 / 30)
HORIZON_CALENDAR_DAYS = 30
FEATURES = ['spread', 'amd_earnings_days_capped', 'nvda_earnings_days_capped', 'trailing_rv_amd', 'trailing_rv_nvda']
NEW_FEATURES = FEATURES + ['rv_adjusted_value', 'spread_reversion']
LADDER = np.arange(-1, 1.01, 0.25)

def payoff_sharpe(reward):
    """Annualized 30-day payoff Sharpe; all-flat scores zero by convention.

    Square-root-of-time convention for the payoff horizon, not the daily entry
    frequency. This is not a serial-correlation-adjusted portfolio Sharpe.
    """
    reward = np.asarray(reward, dtype=float)
    sd = reward.std(ddof=1)
    return float(SHARPE_ANNUALIZATION * reward.mean() / sd) if sd > 1e-12 else 0.0

def net_rewards(actions, reward, cost):
    return actions * reward - cost * np.abs(actions)

def policy_scores(policies, state, reward, nstates, cost=DEFAULT_EXECUTION_COST):
    """Exact full-episode Sharpe using per-state sufficient statistics."""
    n = len(reward)
    sums = np.bincount(state, weights=reward, minlength=nstates)
    squares = np.bincount(state, weights=reward ** 2, minlength=nstates)
    counts = np.bincount(state, minlength=nstates)
    squared = policies ** 2
    magnitude = np.abs(policies)
    if np.ndim(cost) == 0:
        mean = (policies @ sums - cost * (magnitude @ counts)) / n
        second = squared @ squares - 2 * cost * (policies * magnitude @ sums) + cost ** 2 * (squared @ counts)
    else:
        costs = np.asarray(cost, dtype=float)
        if costs.shape != np.shape(reward) or not np.isfinite(costs).all() or (costs < 0).any():
            raise ValueError('Costs must be finite nonnegative values aligned with rewards')
        cost_sums = np.bincount(state, weights=costs, minlength=nstates)
        cross = np.bincount(state, weights=costs * reward, minlength=nstates)
        cost_squares = np.bincount(state, weights=costs ** 2, minlength=nstates)
        mean = (policies @ sums - magnitude @ cost_sums) / n
        second = squared @ squares - 2 * (policies * magnitude @ cross) + squared @ cost_squares
    var = np.maximum((second - n * mean ** 2) / (n - 1), 0)
    return SHARPE_ANNUALIZATION * np.divide(mean, np.sqrt(var), out=np.zeros_like(mean), where=var > 1e-24)

def fit_policy(state, reward, nstates, seed, cost=DEFAULT_EXECUTION_COST, iterations=80, population=256, action_values=None, initial_policy=None):
    """Cross-entropy episodic policy search maximizing training payoff Sharpe.

    An episode replays the training history; its terminal objective is Sharpe.
    Candidate policies map each IV-spread bucket to a fixed action. No future
    outcomes enter their observations. This is direct policy search, not TD Q-learning.
    """
    rng = np.random.default_rng(seed)
    action_values = ACTIONS if action_values is None else np.asarray(action_values, dtype=float)
    if action_values.ndim != 1 or not np.isfinite(action_values).all() or 0 not in action_values or (len(np.unique(action_values)) != len(action_values)):
        raise ValueError('Action values must be finite, unique, and include flat')
    n_actions = len(action_values)
    probabilities = np.full((nstates, n_actions), 1 / n_actions)
    best = np.zeros(nstates, dtype=float)
    best_score = 0.0
    if initial_policy is not None:
        initial_policy = np.asarray(initial_policy, dtype=float)
        if initial_policy.shape != (nstates,) or not np.isin(initial_policy, action_values).all():
            raise ValueError('Initial policy must use the specified ladder')
        initial_score = float(policy_scores(initial_policy[None, :], state, reward, nstates, cost)[0])
        if initial_score > best_score:
            best = initial_policy.copy()
            best_score = initial_score
    for _ in range(iterations):
        draws = rng.random((population, nstates, 1))
        indices = (draws > probabilities.cumsum(axis=1)[None, :, :]).sum(axis=2)
        indices = np.minimum(indices, n_actions - 1)
        policies = action_values[indices]
        policies[0] = best
        scores = policy_scores(policies, state, reward, nstates, cost)
        winner = int(scores.argmax())
        if scores[winner] > best_score:
            best_score = float(scores[winner])
            best = policies[winner].copy()
        elite = policies[np.argsort(scores)[-max(2, population // 10):]]
        frequencies = np.stack([(elite == a).mean(axis=0) for a in action_values], axis=1)
        probabilities = 0.3 * probabilities + 0.7 * frequencies
        probabilities = 0.97 * probabilities + 0.03 / n_actions
    best[np.bincount(state, minlength=nstates) == 0] = 0
    return (best, best_score)

def entry_cost(panel, rate):
    return np.ones(len(panel)) if rate is None else rate * (panel.iv_amd + panel.iv_nvda).to_numpy(float) / 2

def leaf_states(tree, data, features, leaf_ids):
    leaves = tree.apply(data[features])
    return np.searchsorted(leaf_ids, leaves)

def fold_data(panel, test_year):
    validation_start = pd.Timestamp(test_year - 2, 1, 1)
    test_start = pd.Timestamp(test_year, 1, 1)
    test_end = pd.Timestamp(test_year + 2, 1, 1)
    train = panel[(panel.index < validation_start) & (panel.end < validation_start)]
    validation = panel[(panel.index >= validation_start) & (panel.index < test_start) & (panel.end < test_start)]
    test = panel[(panel.index >= test_start) & (panel.index < test_end)]
    return (train, validation, test)

def eligible_fold(train, val, test):
    return len(train) >= 504 and len(val) >= 252 and (len(test) >= 126)

def add_features(panel):
    p = panel.copy()
    past = p.spread.shift(1).rolling(252, min_periods=126)
    p['rv_adjusted_value'] = p.trailing_rv_amd - p.iv_amd - (p.trailing_rv_nvda - p.iv_nvda)
    p['spread_reversion'] = (past.mean() - p.spread) / past.std().replace(0, np.nan)
    return p.replace([np.inf, -np.inf], np.nan).dropna(subset=NEW_FEATURES)

def nn_actions(prediction, cost, multiplier):
    prediction = np.asarray(prediction)
    return np.where(np.abs(prediction) > multiplier * np.asarray(cost), np.sign(prediction), 0.0)

def fit_nn(train, validation, test):
    """One hidden layer; training-only scalers, validation-only selection."""
    scaler = StandardScaler().fit(train[NEW_FEATURES])
    xtr = scaler.transform(train[NEW_FEATURES])
    xv = scaler.transform(validation[NEW_FEATURES])
    xt = scaler.transform(test[NEW_FEATURES])
    y = train.long_reward.to_numpy()
    center = y.mean()
    scale = max(y.std(), 1e-08)
    candidates = []
    models = {}
    for alpha in [0.01, 1.0]:
        model = MLPRegressor(hidden_layer_sizes=(16,), activation='tanh', solver='lbfgs', alpha=alpha, max_iter=10000, max_fun=200000, tol=0.0001, random_state=2026)
        with warnings.catch_warnings(record=True) as messages:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(xtr, (y - center) / scale)
        converged = not any((issubclass(w.category, ConvergenceWarning) for w in messages))
        vp = model.predict(xv) * scale + center
        models[alpha] = model
        for multiplier in [1.0, 2.0, 4.0]:
            action = nn_actions(vp, entry_cost(validation, 0.025), multiplier)
            score = payoff_sharpe(net_rewards(action, validation.long_reward.to_numpy(), entry_cost(validation, 0.025)))
            candidates.append(dict(model='nn_both_fixed', alpha=alpha, threshold=multiplier, validation_sharpe=score, converged=converged, iterations=model.n_iter_))
    chosen = max(candidates, key=lambda c: c['validation_sharpe'])
    model = models[chosen['alpha']]
    action = nn_actions(model.predict(xt) * scale + center, entry_cost(test, 0.025), chosen['threshold'])
    return (action, chosen, candidates)

def fit_trees(train, validation, test, features, include_fixed):
    candidates = []
    fits = []
    trcost = entry_cost(train, 0.025)
    vacost = entry_cost(validation, 0.025)
    for leaves in [4, 8, 16]:
        for minimum in [126, 252]:
            tree = DecisionTreeRegressor(max_leaf_nodes=leaves, min_samples_leaf=minimum, random_state=2026)
            tree.fit(train[features], train.long_reward)
            ids = np.unique(tree.apply(train[features]))
            states = leaf_states(tree, train, features, ids)
            (fixed, fs) = max([fit_policy(states, train.long_reward.to_numpy(), len(ids), seed, cost=trcost) for seed in range(5)], key=lambda p: p[1])
            (sized, ss) = max([fit_policy(states, train.long_reward.to_numpy(), len(ids), seed, cost=trcost, action_values=LADDER, initial_policy=fixed) for seed in range(5)], key=lambda p: p[1])
            assert ss >= fs - 1e-10
            for (kind, policy, score) in [('sized', sized, ss)] + ([('fixed', fixed, fs)] if include_fixed else []):
                va = policy[leaf_states(tree, validation, features, ids)]
                cfg = dict(model=('rl_both_' if include_fixed else 'rl_base_') + kind, leaves=leaves, min_leaf=minimum, train_sharpe=score, validation_sharpe=payoff_sharpe(net_rewards(va, validation.long_reward.to_numpy(), vacost)))
                candidates.append(cfg)
                fits.append((cfg, tree, ids, policy))
    selected = {}
    for name in dict.fromkeys((c['model'] for c in candidates)):
        (cfg, tree, ids, policy) = max([f for f in fits if f[0]['model'] == name], key=lambda f: f[0]['validation_sharpe'])
        selected[name] = (policy[leaf_states(tree, test, features, ids)], cfg)
    return (selected, candidates)

def parameter_count(hidden):
    return 7 * hidden + hidden + hidden * 9 + 9

def logits(weights, x, hidden):
    """Batched networks: population x observations x nine actions."""
    w = np.atleast_2d(weights)
    k = 7 * hidden
    h = np.tanh(np.einsum('nd,pdh->pnh', x, w[:, :k].reshape(-1, 7, hidden)) + w[:, k:k + hidden, None].transpose(0, 2, 1))
    k += hidden
    return np.einsum('pnh,pha->pna', h, w[:, k:k + hidden * 9].reshape(-1, hidden, 9)) + w[:, -9:, None].transpose(0, 2, 1)

def probabilities(weights, x, hidden):
    z = logits(weights, x, hidden)
    z -= z.max(axis=-1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(axis=-1, keepdims=True)

def actions(weights, x, hidden):
    return LADDER[logits(weights, x, hidden).argmax(axis=-1)]

def exact_scores(action, reward, cost):
    r = net_rewards(action, reward, cost)
    sd = r.std(axis=-1, ddof=1)
    return SHARPE_ANNUALIZATION * np.divide(r.mean(axis=-1), sd, out=np.zeros_like(sd), where=sd > 1e-12)

def search(x, reward, cost, hidden, seed, iterations=80, population=256, batch=16):
    """Diagonal Gaussian CEM; rank candidates by exact full-history Sharpe.

    Smoothing and exploration floor regularize the search distribution, not
    the objective. The incumbent is retained; all-flat has score zero.
    """
    rng = np.random.default_rng(seed)
    dim = parameter_count(hidden)
    mean = np.zeros(dim)
    std = np.full(dim, 0.5)
    best = mean.copy()
    best[-5] = 1.0
    best_score = 0.0
    history = []
    for _ in range(iterations):
        population_weights = rng.normal(mean, std, size=(population, dim))
        population_weights[0] = best
        scores = np.concatenate([exact_scores(actions(population_weights[j:j + batch], x, hidden), reward, cost) for j in range(0, population, batch)])
        winner = scores.argmax()
        if scores[winner] > best_score:
            best_score = float(scores[winner])
            best = population_weights[winner].copy()
        elite = population_weights[np.argsort(scores)[-max(2, population // 10):]]
        mean = 0.3 * mean + 0.7 * elite.mean(axis=0)
        std = np.maximum(0.3 * std + 0.7 * elite.std(axis=0), 0.03)
        history.append(best_score)
    return (best, best_score, history)

def fit_neural(train, validation, *, iterations=80, population=256, seeds=5, widths=(4, 8, 16), batch=16):
    """Select seed on training only and width on validation only; no test input."""
    scaler = StandardScaler().fit(train[NEW_FEATURES])
    x = scaler.transform(train[NEW_FEATURES])
    xv = scaler.transform(validation[NEW_FEATURES])
    candidates = []
    fits = []
    for hidden in widths:
        trials = []
        for seed in range(seeds):
            trials.append(search(x, train.long_reward.to_numpy(), entry_cost(train, 0.025), hidden, seed, iterations, population, batch))
            print(f'Neural width {hidden}, seed {seed + 1}/{seeds} finished', flush=True)
        winner = max(range(seeds), key=lambda s: trials[s][1])
        (weights, score, history) = trials[winner]
        va = actions(weights, xv, hidden)[0]
        cfg = dict(hidden=hidden, seed=winner, train_sharpe=score, validation_sharpe=payoff_sharpe(net_rewards(va, validation.long_reward.to_numpy(), entry_cost(validation, 0.025))), seed_train_scores=[t[1] for t in trials], training_history=history)
        candidates.append(cfg)
        fits.append((weights, cfg))
    (weights, cfg) = max(fits, key=lambda f: f[1]['validation_sharpe'])
    return (scaler, weights, cfg, candidates)

def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--data', type=Path, default=Path('data/current_dataset.csv'), help='Single-pair CSV with the existing panel schema')
    ap.add_argument('--pair', default='AMD_NVDA')
    ap.add_argument('--years', type=int, nargs='+', default=list(range(2010, 2026, 2)))
    ap.add_argument('--output', type=Path, default=Path('artifacts/neural_policy'))
    ap.add_argument('--iterations', type=int, default=80)
    ap.add_argument('--population', type=int, default=256)
    ap.add_argument('--seeds', type=int, default=5)
    ap.add_argument('--widths', type=int, nargs='+', default=[4, 8, 16])
    ap.add_argument('--batch', type=int, default=16)
    ap.add_argument('--smoke', action='store_true', help='One fold, tiny neural budget; skip expensive reference fits')
    args = ap.parse_args()
    if args.smoke:
        args.years = [args.years[0]]
        (args.iterations, args.population, args.seeds, args.widths) = (2, 16, 1, [4])
    if min(args.iterations, args.seeds, args.batch, *args.widths) < 1 or args.population < 2:
        ap.error('Positive dimensions/budgets and population >= 2 required')
    if args.output.exists() and any(args.output.iterdir()):
        ap.error('Choose an empty output directory to avoid mixing runs')
    args.output.mkdir(parents=True, exist_ok=True)
    settings = {k: str(v) if isinstance(v, Path) else v for (k, v) in vars(args).items()}
    settings.update(features=NEW_FEATURES, ladder=LADDER.tolist(), cost_rate=0.025, annualization=float(SHARPE_ANNUALIZATION), execution='softmax argmax', source_sha256=hashlib.sha256(args.data.read_bytes()).hexdigest(), code_sha256={Path(__file__).name: hashlib.sha256(Path(__file__).read_bytes()).hexdigest()})
    (args.output / 'settings.json').write_text(json.dumps(settings, indent=2))
    raw = pd.read_csv(args.data, index_col='date', parse_dates=['date', 'end']).sort_index()
    if raw.index.has_duplicates:
        raise ValueError('Duplicate entry dates')
    raw['long_reward'] = raw.rv_amd - raw.iv_amd - (raw.rv_nvda - raw.iv_nvda)
    panel = add_features(raw)
    (rows, configs) = ([], [])
    start = time.monotonic()
    for year in args.years:
        (tr, va, te) = fold_data(panel, year)
        if not eligible_fold(tr, va, te):
            print(f'Skipping ineligible fold {year}', flush=True)
            continue
        assert tr.end.max() < va.index.min() and va.end.max() < te.index.min()
        assert te.index.equals(fold_data(raw, year)[2].index)
        for frame in (tr, va, te):
            if not np.isfinite(frame[NEW_FEATURES + ['long_reward', 'iv_amd', 'iv_nvda']].to_numpy()).all():
                raise ValueError('Nonfinite inputs or outcomes')
        t = time.monotonic()
        (scaler, weights, cfg, candidates) = fit_neural(tr, va, iterations=args.iterations, population=args.population, seeds=args.seeds, widths=args.widths, batch=args.batch)
        neural_seconds = time.monotonic() - t
        print(f'Neural fit finished in {neural_seconds:.1f}s; fitting tree reference next', flush=True)
        act = actions(weights, scaler.transform(te[NEW_FEATURES]), cfg['hidden'])[0]
        np.savez(args.output / f'{year}_policy.npz', weights=weights, mean=scaler.mean_, scale=scaler.scale_, hidden=cfg['hidden'])
        t = time.monotonic()
        (reference, tree_candidates) = ({}, []) if args.smoke else fit_trees(tr, va, te, NEW_FEATURES, True)
        tree_seconds = time.monotonic() - t
        print(f'Tree fit finished in {tree_seconds:.1f}s; fitting forecast baseline next', flush=True)
        t = time.monotonic()
        old_candidates = []
        if not args.smoke:
            (old_actions, old_cfg, old_candidates) = fit_nn(tr, va, te)
            reference['nn_both_fixed'] = (old_actions, old_cfg)
        old_seconds = time.monotonic() - t
        reference['neural_sized'] = (act, cfg)
        result = te[['end']].copy()
        result['fold'] = year
        for (name, (a, _)) in reference.items():
            assert np.isin(a, LADDER).all()
            result[name + '_action'] = a
            result[name + '_reward'] = net_rewards(a, te.long_reward.to_numpy(), entry_cost(te, 0.025))
        result.to_csv(args.output / f'{year}.csv')
        rows.append(result)
        configs.append(dict(fold=year, train_entries=len(tr), validation_entries=len(va), test_entries=len(te), train_last_outcome=str(tr.end.max()), validation_first=str(va.index.min()), validation_last_outcome=str(va.end.max()), test_first=str(te.index.min()), neural_seconds=neural_seconds, tree_seconds=tree_seconds, old_nn_seconds=old_seconds, selected={n: c for (n, (_, c)) in reference.items()}, candidates=dict(neural=candidates, tree=tree_candidates, old_nn=old_candidates)))
        (args.output / 'configs.json').write_text(json.dumps(configs, indent=2))
        print(f'Completed {year}: neural {neural_seconds:.1f}s, tree {tree_seconds:.1f}s, forecast {old_seconds:.1f}s', flush=True)
    if not rows:
        raise ValueError('No eligible folds')
    pred = pd.concat(rows).sort_index()
    if pred.index.has_duplicates:
        raise ValueError('Test folds overlap; use disjoint two-year starts')
    pred.to_csv(args.output / 'predictions.csv')
    scores = [dict(pair=args.pair, model=n, entries=len(pred), sharpe=payoff_sharpe(pred[n + '_reward']), net_pnl=pred[n + '_reward'].sum(), mean_absolute_size=pred[n + '_action'].abs().mean()) for n in reference]
    pd.DataFrame(scores).to_csv(args.output / 'scores.csv', index=False)
    (args.output / 'runtime.json').write_text(json.dumps(dict(seconds=time.monotonic() - start)))
    print(pd.DataFrame(scores).to_string(index=False))

if __name__ == '__main__':
    main()


In [ ]:
import subprocess, sys, time, json, zipfile
from pathlib import Path
from google.colab import files
output = Path('/content') / ('neural_results_' + str(time.time_ns()))
output.mkdir()
status = {'state': 'running', 'scope': 'AMD/NVDA cached-panel matched comparison'}
try:
    with (output / 'run.log').open('w') as log:
        with subprocess.Popen([sys.executable, '-u', '/content/train_neural.py',
                '--data', 'data/current_dataset.csv', '--output', str(output / 'benchmark')],
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                if process.wait():
                    raise RuntimeError('Training failed; return the ZIP for diagnosis')
            except BaseException:
                process.terminate()
                process.wait()
                raise
    status['state'] = 'complete'
except BaseException as exc:
    status.update(state='incomplete', error=str(exc))
    raise
finally:
    (output / 'status.json').write_text(json.dumps(status, indent=2))
    result = Path('/content/neural_policy_results.zip')
    with zipfile.ZipFile(result, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in output.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to(output))
        archive.write('/content/train_neural.py', 'train_neural.py')
        archive.write('data/current_dataset.provenance.json', 'data_provenance.json')
    print('Attach neural_policy_results.zip to the chat.')
    files.download(str(result))
